# ATP Tennis Analysis

This notebook evaluates the three research goals:

1. How important is ranking for match outcomes?
2. Do players perform differently depending on the surface?
3. Which player characteristics are associated with winning matches?

The analysis uses:
- `data/matches_clean.csv`
- `data/stats_clean.csv`
- `data/top20_players_raw.csv`

In [35]:
import pandas as pd
import numpy as np

# import of CSV files
matches = pd.read_csv("data/matches_clean.csv")
stats = pd.read_csv("data/stats_clean.csv")
players = pd.read_csv("data/top20_players_raw.csv")

## Data quality inspection
Before proceeding to the main analysis, we inspect the cleaned data for any remaining issues:
- verify the number of missing values
- check for duplicate matches
- identify any rows with incomplete information

This ensures the dataset is reliable for the research questions.

In [36]:
# Basic data checks
print("\nMatches columns:", matches.columns.tolist())
print("Stats columns:", stats.columns.tolist())
print("Players columns:", players.columns.tolist())

print("\nMatches dtypes:")
print(matches.dtypes.to_string())

print("\nMissing values in matches:")
print(matches.isna().sum())


Matches columns: ['date', 'tournament', 'surface', 'Round', 'winner', 'winner_rank_at_match', 'loser', 'loser_rank_at_match', 'score', 'source_player']
Stats columns: ['player', 'surface', 'match_record', 'match_wins', 'match_losses', 'match_pct', 'tiebreak_record', 'tiebreak_wins', 'tiebreak_losses', 'tiebreak_pct', 'ace_rate_pct', 'first_serve_in_pct', 'first_serve_points_won_pct', 'second_serve_points_won_pct', 'service_games_held_pct', 'service_points_won_pct', 'return_games_won_pct', 'return_points_won_pct', 'total_points_won_pct', 'dominance_ratio']
Players columns: ['name', 'age', 'current_rank']

Matches dtypes:
date                        str
tournament                  str
surface                     str
Round                       str
winner                      str
winner_rank_at_match    float64
loser                       str
loser_rank_at_match     float64
score                       str
source_player               str

Missing values in matches:
date                   

In [38]:
missing = matches[matches["winner_rank_at_match"].isna() | matches["loser_rank_at_match"].isna()]

print("Total missing rows:", len(missing))
print(missing[["date", "source_player", "winner", "loser", "score", "winner_rank_at_match", "loser_rank_at_match"]].head())

print("\nCounts by source_player:")
print(missing["source_player"].value_counts())

Total missing rows: 1
           date     source_player            winner     loser    score  \
712  6-Feb-2026  Alexander Bublik  Alexander Bublik  Hugo Nys  6-0 6-3   

     winner_rank_at_match  loser_rank_at_match  
712                  10.0                  NaN  

Counts by source_player:
source_player
Alexander Bublik    1
Name: count, dtype: int64


In [39]:
print("\nStats dtypes:")
print(stats.dtypes.to_string())


Stats dtypes:
player                             str
surface                            str
match_record                       str
match_wins                       int64
match_losses                     int64
match_pct                          str
tiebreak_record                    str
tiebreak_wins                    int64
tiebreak_losses                  int64
tiebreak_pct                       str
ace_rate_pct                       str
first_serve_in_pct                 str
first_serve_points_won_pct         str
second_serve_points_won_pct        str
service_games_held_pct             str
service_points_won_pct             str
return_games_won_pct               str
return_points_won_pct              str
total_points_won_pct               str
dominance_ratio                float64


In [40]:
print("\nMissing values in stats:")
print(stats.isna().sum())


Missing values in stats:
player                         0
surface                        0
match_record                   0
match_wins                     0
match_losses                   0
match_pct                      0
tiebreak_record                0
tiebreak_wins                  0
tiebreak_losses                0
tiebreak_pct                   0
ace_rate_pct                   1
first_serve_in_pct             1
first_serve_points_won_pct     1
second_serve_points_won_pct    1
service_games_held_pct         1
service_points_won_pct         1
return_games_won_pct           1
return_points_won_pct          1
total_points_won_pct           1
dominance_ratio                1
dtype: int64


In [41]:
with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.width', None, 'display.expand_frame_repr', False):
    print(stats[stats.isna().any(axis=1)].to_string())

         player surface match_record  match_wins  match_losses match_pct tiebreak_record  tiebreak_wins  tiebreak_losses tiebreak_pct ace_rate_pct first_serve_in_pct first_serve_points_won_pct second_serve_points_won_pct service_games_held_pct service_points_won_pct return_games_won_pct return_points_won_pct total_points_won_pct  dominance_ratio
47  Casper Ruud   Grass      0-0 (-)           0             0         0         0-0 (-)              0                0            0          NaN                NaN                        NaN                         NaN                    NaN                    NaN                  NaN                   NaN                  NaN              NaN


In [42]:
print("\nDuplicate match rows:", matches.duplicated(
   subset=["date", "tournament", "surface", "Round", "winner", "loser", "score"]
).sum())


Duplicate match rows: 133


In [43]:
# Inspect whether duplicates are exact copies, just from different sources (2 players)
duplicates = matches[matches.duplicated(subset=["date", "tournament", "surface", "Round", "winner", "loser", "score"], keep=False)]

exact_duplicate_counts = duplicates.groupby(
    ["date", "tournament", "surface", "Round", "winner", "loser", "score"]
).size().sort_values(ascending=False).tail(30)

print("\nTop duplicate groups by count:")
print(exact_duplicate_counts)


Top duplicate groups by count:
date        tournament            surface  Round  winner            loser                score         
4-Mar-2026  Indian Wells Masters  Hard     SF     Jannik Sinner     Alexander Zverev     6-2 6-4           2
5-Apr-2026  Monte-Carlo           Clay     R16    Alexander Bublik  Jiri Lehecka         6-2 7-5           2
5-Jan-2026  Hong Kong             Hard     F      Alexander Bublik  Lorenzo Musetti      7-6(2) 6-3        2
                                           SF     Lorenzo Musetti   Andrey Rublev        6-7(3) 7-5 6-4    2
7-Aug-2025  Cincinnati Masters    Hard     F      Carlos Alcaraz    Jannik Sinner        5-0 RET           2
                                           QF     Alexander Zverev  Ben Shelton          6-2 6-2           2
                                                  Carlos Alcaraz    Andrey Rublev        6-3 4-6 7-5       2
                                           R16    Alexander Zverev  Karen Khachanov      7-5 3-0 RET 

### Note
1. Missing rank values in matches: loser was unranked at match time
2. Missing stat values (NaN) in stats: player never played on that surface (grass)
3. Duplicate matches: same match scraped twice, once per player perspective

Actions taken:
- Duplicates removed keeping first occurrence
- TODO: parse percentage columns from string (e.g. "79.8%") to float


In [44]:
matches = matches.drop_duplicates(subset=["date", "tournament", "surface", "Round", "winner", "loser", "score"], keep="first")
# Use the deduplicated version for analysis

In [45]:
# Parse stats percentages
percentage_columns = [
    "match_pct", "tiebreak_pct", "ace_rate_pct", "first_serve_in_pct", "first_serve_points_won_pct",
    "second_serve_points_won_pct", "service_games_held_pct", "service_points_won_pct",
    "return_games_won_pct", "return_points_won_pct", "total_points_won_pct",
    "dominance_ratio"
]

for col in percentage_columns:
    if col in stats.columns:
        stats[col] = pd.to_numeric(
            stats[col].astype(str).str.rstrip("%"),
            errors="coerce",
        ).astype("float64")

stats.head(5)

,player,surface,match_record,match_wins,match_losses,match_pct,tiebreak_record,tiebreak_wins,tiebreak_losses,tiebreak_pct,ace_rate_pct,first_serve_in_pct,first_serve_points_won_pct,second_serve_points_won_pct,service_games_held_pct,service_points_won_pct,return_games_won_pct,return_points_won_pct,total_points_won_pct,dominance_ratio
0,Carlos Alcaraz,Overall last 52,70-7 (91%),70,7,91.0,20-13 (61%),20,13,61.0,7.5,65.5,74.7,57.1,89.1,68.6,30.1,41.4,54.5,1.32
1,Carlos Alcaraz,Hard,40-5 (89%),40,5,89.0,10-7 (59%),10,7,59.0,7.7,66.1,75.9,57.6,90.6,69.7,30.0,41.2,54.7,1.36
2,Carlos Alcaraz,Clay,19-1 (95%),19,1,95.0,8-2 (80%),8,2,80.0,3.5,67.2,69.5,56.7,84.1,65.3,36.2,44.7,54.7,1.29
3,Carlos Alcaraz,Grass,11-1 (92%),11,1,92.0,2-4 (33%),2,4,33.0,12.4,61.6,78.9,56.5,91.4,70.3,22.6,37.5,53.6,1.26
4,Jannik Sinner,Overall last 52,72-8 (90%),72,8,90.0,19-5 (79%),19,5,79.0,10.8,63.7,79.8,58.2,92.2,71.9,32.3,42.2,56.3,1.51


## Data validation

We begin by checking:
- verify no duplicate match rows remain
- confirm stats percentage fields are floats

In [46]:
print("\nDuplicate rows after validation:", matches.duplicated(
    subset=["date", "tournament", "surface", "Round", "winner", "loser", "score"]
).sum())
print("\nStats dtypes:")
print(stats.dtypes.to_string())


Duplicate rows after validation: 0

Stats dtypes:
player                             str
surface                            str
match_record                       str
match_wins                       int64
match_losses                     int64
match_pct                      float64
tiebreak_record                    str
tiebreak_wins                    int64
tiebreak_losses                  int64
tiebreak_pct                   float64
ace_rate_pct                   float64
first_serve_in_pct             float64
first_serve_points_won_pct     float64
second_serve_points_won_pct    float64
service_games_held_pct         float64
service_points_won_pct         float64
return_games_won_pct           float64
return_points_won_pct          float64
total_points_won_pct           float64
dominance_ratio                float64
